# DAS QuakeMigrate Glacier Example

This example shows how to process fibreoptic and seismic node data for near-surface icequakes. Included here are all the data neccessary for running the example, along with a yml file containing the install environment used to run the example.

In [1]:
# Import neccessary modules:
import pandas as pd
from obspy.core import AttribDict
from pyproj import Proj
from quakemigrate import QuakeScan, Trigger
from quakemigrate.io import Archive, read_lut, read_stations, read_vmodel
from quakemigrate.lut import compute_traveltimes
from quakemigrate.signal.onsets import STALTAOnset
from quakemigrate.signal.pickers import GaussianPicker


/var/folders/hg/xp5sp5p14knc98zsdqx2p7cw0000gn/T/ipykernel_54919/2642654472.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## 0. Specify i/o paths:

In [2]:
# --- i/o paths ---
station_file = "inputs/gornerglethscer_das_and_nodes_qm_stations.csv" 
data_in = "inputs/data/nodes"
das_data_in = "inputs/data/das"
lut_out = "outputs/lut/gornergletscher_das_and_nodes_no_DASsens_sparser.LUT" 
run_path = "outputs/runs"
run_name = "gorner_das_and_nodes_run_EXAMPLE_EVENT"

# --- Set time period over which to run detect ---
starttime = "2023-10-22T06:26:00.000000Z" 
endtime = "2023-10-22T06:26:10.000000Z"  



## 1. Create the travel-time lookup table

(Note: We do not create a sensitivity lookup table here, but one is easily created using the commented out line below. This is only emitted as it takes time to calculate all the ray paths).

In [3]:
# --- Read in the station information file ---
stations = read_stations(station_file)

# --- Define the input and grid projections ---
gproj = Proj(proj="lcc", units="km", lon_0=7.817, lat_0=45.9793, lat_1=45.979,
             lat_2=45.9796, datum="WGS84", ellps="WGS84", no_defs=True)
cproj = Proj(proj="longlat", datum="WGS84", ellps="WGS84", no_defs=True)

# --- Define the grid specifications ---
# The ObsPy AttribDict behaves like a Python dict, but with '.'-style access.
grid_spec = AttribDict()
grid_spec.ll_corner = [7.8155, 45.9785, -2.760]
grid_spec.ur_corner = [7.8180, 45.980, -2.600]
grid_spec.node_spacing = [0.008, 0.008, 0.01] #[0.003, 0.003, 0.006] 
grid_spec.grid_proj = gproj
grid_spec.coord_proj = cproj

# --- Homogeneous LUT generation ---
lut = compute_traveltimes(grid_spec, stations, method="homogeneous",
                            phases=["P", "S"], log=True, vp=3.84, vs=1.71,
                            save_file=lut_out)

# # --- And calculate DAS sensitivities and update LUT ---
# lut = read_lut(lut_file=lut_out_orig)
# compute_das_sensitivity(lut, grid_spec, station_prefix="D",
#                         das_sens_lower_cutoff=0.1, 
#                         write_out_das_sensitivities=True,
#                         save_file=lut_out, log=True)

Computing homogeneous traveltimes for...
	...phase: P...
		...station: N01 - 1 of 426
		...station: N02 - 2 of 426
		...station: N03 - 3 of 426
		...station: N04 - 4 of 426
		...station: N05 - 5 of 426
		...station: N06 - 6 of 426
		...station: N07 - 7 of 426
		...station: N08 - 8 of 426
		...station: N09 - 9 of 426
		...station: N10 - 10 of 426
		...station: N11 - 11 of 426
		...station: N12 - 12 of 426
		...station: N13 - 13 of 426
		...station: N14 - 14 of 426
		...station: N15 - 15 of 426
		...station: N17 - 16 of 426
		...station: N18 - 17 of 426
		...station: N19 - 18 of 426
		...station: N20 - 19 of 426
		...station: N21 - 20 of 426
		...station: N22 - 21 of 426
		...station: N23 - 22 of 426
		...station: N24 - 23 of 426
		...station: N25 - 24 of 426
		...station: N26 - 25 of 426
		...station: N27 - 26 of 426
		...station: N28 - 27 of 426
		...station: N29 - 28 of 426
		...station: D0379 - 29 of 426
		...station: D0381 - 30 of 426
		...station: D0382 - 31 of 426
		...station: D0

## 2. Run Detect stage:

(To produce initial coalescence time-series with which to then detect events from).


In [4]:
# --- Read in station file ---
stations = read_stations(station_file)

# --- Set station weights ---
station_phase_weights = {}
for idx, row in stations.iterrows():
    station_phase_weights[row['Name']] = {}
    station_phase_weights[row['Name']]['P'] = 1. 
    station_phase_weights[row['Name']]['S'] = 1. 

# --- Load the LUT ---
lut = read_lut(lut_file=lut_out)

# --- Create new Archive and set path structure ---
archive = Archive(archive_path=data_in, stations=stations,
                  archive_format="YEAR/JD/*_STATION_*",
                  das_archive_path=das_data_in, das_data_fmt="sgy",
                  duplicate_das_comps=True,
                  spatial_down_samp_factor=5,
                  first_last_das_channels=[236, 634],
                  semblance_stack=False,
                  convert_strainrate_to_vel=True,
                  strain_vs_strainrate="strain",
                  channel_spacing=1.6,
                  gauge_length=6.38) #gauge_length=3.19)

# --- Create new Onset ---
onset = STALTAOnset(position="recursive", sampling_rate=1000)
onset.phases = ["P"] 
onset.bandpass_filters = {
    "P": [10, 250, 4],
    "S": [5, 100, 4]}
onset.sta_lta_windows = {
    "P": [0.01, 0.2],
    "S": [0.02, 0.2]}
onset.station_phase_weights = station_phase_weights

# --- Create new QuakeScan ---
scan = QuakeScan(archive, lut, onset=onset, run_path=run_path,
                 run_name=run_name, log=True, loglevel="info")

# --- Set detect parameters ---
scan.timestep = 30. #30. #10.
# NOTE: please increase the thread-count as your system allows; the
# core migration routines are compiled against OpenMP, and using
# multithreading will ~ linearly speed up the compute time!
scan.threads = 8

# --- Run detect ---
scan.detect(starttime, endtime)

	QuakeMigrate RUN - Path: outputs/runs/gorner_das_and_nodes_run_EXAMPLE_EVENT - Name: gorner_das_and_nodes_run_EXAMPLE_EVENT

	DETECT - Continuous coalescence scan

	Scanning from 2023-10-22T06:26:00.000000Z to 2023-10-22T06:26:10.000000Z

	Scan parameters:
		Scan sampling rate = 1000 Hz
		Thread count       = 8
		Time step          = 30.0 s

	Onset parameters - using the recursive STA/LTA onset
		Onset function sampling rate = 1000 Hz
		Phase(s) = ['P']

		P bandpass filter  = [10, 250, 4] (Hz, Hz, -)
		S bandpass filter  = [5, 100, 4] (Hz, Hz, -)

		P onset [STA, LTA] = [0.01, 0.2] (s, s)
		S onset [STA, LTA] = [0.02, 0.2] (s, s)

~~~~~~~~~~~~~~~~~~~~ Processing : 2023-10-22T06:26:00.000000Z-2023-10-22T06:26:30.000000Z ~~~~~~~~~~~~~~~~~~~~
Converting das strain-rate to velocity.
Converting das strain-rate to velocity.
		No P onset for D0379.
		No P onset for D0381.
		No P onset for D0382.
		No P onset for D0384.
		No P onset for D0387.
		No P onset for D0389.
		No P onset for D0390.


## 3. Run Trigger stage:

(Trigger events from outputs from detect stage).


In [5]:
# --- Load the LUT ---
lut = read_lut(lut_file=lut_out)

# --- Create new Trigger ---
trig = Trigger(lut, run_path=run_path, run_name=run_name, log=True,
               loglevel="info")

# --- Set trigger parameters ---
trig.marginal_window = 0.25 #0.05 #0.1 #0.5
trig.min_event_interval = 0.5 #0.1 #0.2 #1. 
trig.normalise_coalescence = True

# --- Static threshold ---
trig.threshold_method = "static"
trig.static_threshold = 1.35 #1.4

# --- Run trigger ---
trig.trigger(starttime, endtime, interactive_plot=False)

	QuakeMigrate RUN - Path: outputs/runs/gorner_das_and_nodes_run_EXAMPLE_EVENT - Name: gorner_das_and_nodes_run_EXAMPLE_EVENT

	TRIGGER - Triggering events from .scanmseed

	Triggering events from 2023-10-22T06:26:00.000000Z to 2023-10-22T06:26:10.000000Z

	Trigger parameters:
		Pre/post pad = 120.0 s
		Marginal window = 0.25 s
		Minimum event interval  = 0.5 s

		Triggering from the normalised maximum coalescence trace.

		Trigger threshold method: static
		Static threshold = 1.35

	Reading in .scanmseed...
	    Warning! No .scanmseed data found for pre-pad!
	    Warning! No .scanmseed data found for post-pad!
	    ...from 2023-10-22T06:26:00.000000Z - 2023-10-22T06:26:29.999000Z.

	Triggering events...

		1 event(s) triggered within the specified region between 2023-10-22T06:26:00.000000Z 
		and 2023-10-22T06:26:10.000000Z

	Writing triggered events to file...
                     Elapsed time: 0.001129 seconds.

	Plotting trigger summary...


/Users/tomhudson/Documents/python/git_repositories/QuakeMigrate/quakemigrate/signal/trigger.py:461: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  triggers = pd.concat(
/Users/tomhudson/Documents/python/git_repositories/QuakeMigrate/quakemigrate/signal/trigger.py:528: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  refined_events = pd.concat(


                     Elapsed time: 0.693938 seconds.


## 4. Run the Locate stage:

(Takes events detected in trigger stage and reruns migration algorithm to refine event detection and quantify uncertainties).


In [6]:
# --- Read in station file ---
stations = read_stations(station_file)

# --- Set station weights ---
# (weights DAS data for S only and less than seismometers)
station_phase_weights = {}
for idx, row in stations.iterrows():
    station_phase_weights[row['Name']] = {}
    station_phase_weights[row['Name']]['P'] = 1. #1. #0. #0
    station_phase_weights[row['Name']]['S'] = 1. #0.5

# --- Load the LUT ---
lut = read_lut(lut_file=lut_out)

# --- Create new Archive and set path structure ---
# archive = Archive(archive_path=data_in, stations=stations,
#                   archive_format="YEAR/JD/*_STATION_*")
archive = Archive(archive_path=data_in, stations=stations,
                  archive_format="YEAR/JD/*_STATION_*",
                  das_archive_path=das_data_in, das_data_fmt="sgy",
                  duplicate_das_comps=True,
                  spatial_down_samp_factor=5,
                  first_last_das_channels=[236, 634],
                  semblance_stack=False,
                  convert_strainrate_to_vel=True,
                  strain_vs_strainrate="strain",
                  channel_spacing=1.6,
                  gauge_length=6.38) #gauge_length=3.19)

# --- Create new Onset ---
onset = STALTAOnset(position="classic", sampling_rate=1000)
onset.phases = ["P"]
# onset.bandpass_filters = {
#     "P": [10, 200, 4],
#     "S": [5, 100, 4]}
# onset.sta_lta_windows = {
#     "P": [0.01, 0.2],
#     "S": [0.02, 0.2]}
onset.bandpass_filters = {
    "P": [10, 250, 4],
    "S": [5, 100, 4]}
onset.sta_lta_windows = {
    "P": [0.01, 0.2],
    "S": [0.02, 0.2]}
onset.station_phase_weights = station_phase_weights

# --- Create new PhasePicker ---
picker = GaussianPicker(onset=onset)
picker.plot_picks = True #False #True
picker.threshold_method = "static" # "MAD"
picker.static_pick_threshold = 3.5 #2.

# --- Create new QuakeScan ---
scan = QuakeScan(archive, lut, onset=onset, picker=picker,
                 run_path=run_path, run_name=run_name, log=True,
                 loglevel="info")

# --- Set locate parameters ---
scan.marginal_window = 0.3 #0.1 #0.2 #0.2 #0.2 #0.1 #0.5
# NOTE: please increase the thread-count as your system allows; the
# core migration routines are compiled against OpenMP, and using
# multithreading will ~ linearly speed up the compute time!

scan.threads = 8

# --- Toggle plotting options ---
scan.plot_event_summary = True

# --- Toggle writing of waveforms ---
scan.write_cut_waveforms = False

# --- Run locate ---
scan.locate(starttime=starttime, endtime=endtime)

	QuakeMigrate RUN - Path: outputs/runs/gorner_das_and_nodes_run_EXAMPLE_EVENT - Name: gorner_das_and_nodes_run_EXAMPLE_EVENT

	LOCATE - Determining event location and uncertainty

	Locating events from 2023-10-22T06:26:00.000000Z to 2023-10-22T06:26:10.000000Z

	Scan parameters:
		Scan sampling rate = 1000 Hz
		Thread count       = 8
		Marginal window    = 0.3 s

	Onset parameters - using the classic STA/LTA onset
		Onset function sampling rate = 1000 Hz
		Phase(s) = ['P']

		P bandpass filter  = [10, 250, 4] (Hz, Hz, -)
		S bandpass filter  = [5, 100, 4] (Hz, Hz, -)

		P onset [STA, LTA] = [0.01, 0.2] (s, s)
		S onset [STA, LTA] = [0.02, 0.2] (s, s)

	Phase picking by fitting a 1-D Gaussian to onsets
		Static threshold  = 3.5

	EVENT - 1 of 1 - 20231022062604208
	Reading waveform data...
Converting das strain-rate to velocity.
Converting das strain-rate to velocity.
                     Elapsed time: 7.134926 seconds.
	Computing 4-D coalescence function...
		No P onset for D0379.
		No